In [1]:
import sys, json, logging, time
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd, load_active_stressors

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'

In [2]:
CONFIG = {
    'SEED': 42, 'TEST_SIZE': 0.2,
    'CATBOOST_ITERATIONS': 400, 'CATBOOST_LEARNING_RATE': 0.05, 'CATBOOST_DEPTH': 4
}

In [3]:
labeled_pd = load_labeled_pd(DATA_DIR)
active_stressors = load_active_stressors(DATA_DIR)
test_stressors = labeled_pd[active_stressors].sum().sort_values(ascending=False).head(5).index.tolist()

In [4]:
available = ['aa', 'kmer2', 'kmer3', 'onehot', 'physicochemical', 'blosum62', 'length_moments', #'esm2', 
             'genome_dna']
feature_dfs = {}
for name in available:
    p = DATA_DIR / f"features_{name}.parquet"
    if p.exists():
        df = pd.read_parquet(p).drop(columns=['organism'], errors='ignore')
        feature_dfs[name] = df

In [5]:
# Train/test split (use first stressor for groups)
groups = labeled_pd['organism']
gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['SEED'])
train_idx, test_idx = next(gss.split(labeled_pd, labeled_pd[test_stressors[0]], groups=groups))

In [6]:
COMBINATIONS = [('aa',),('kmer2',),('kmer3',),('onehot',),('physicochemical',),
                ('blosum62',),('length_moments',),#('esm2',),
                ('aa','physicochemical'),('aa','onehot'),('aa','kmer2'),#('aa','esm2'),
                ('physicochemical','blosum62','length_moments'),
                ('aa','physicochemical','onehot','kmer2','kmer3'),
                ('aa','physicochemical','blosum62','length_moments',#'esm2'
                ),
                ('aa','kmer2','onehot','physicochemical','blosum62','length_moments',#'esm2'
                ),
                ('aa','kmer2','onehot','physicochemical','blosum62','length_moments',#'esm2',
                 'genome_dna'),
                ('aa',#'esm2',
                 'genome_dna'),
                ('genome_dna',),]

In [7]:
results = []
for combo in COMBINATIONS:
    combo_name = '+'.join(combo)
    log.info(f"Evaluating {combo_name}")
    X_list = [feature_dfs[n] for n in combo]
    X = pd.concat(X_list, axis=1)
    for s in test_stressors:
        y = labeled_pd[s]
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        if y_te.nunique() < 2: continue
        model = CatBoostClassifier(iterations=CONFIG['CATBOOST_ITERATIONS'],
                                   learning_rate=CONFIG['CATBOOST_LEARNING_RATE'],
                                   depth=CONFIG['CATBOOST_DEPTH'],
                                   cat_features=None, eval_metric='F1',
                                   random_seed=CONFIG['SEED'], verbose=False)
        model.fit(X_tr, y_tr, verbose=False)
        y_pred = model.predict(X_te)
        y_proba = model.predict_proba(X_te)[:, 1]
        results.append({'combination': combo_name, 'stressor': s,
                        'mcc': matthews_corrcoef(y_te, y_pred),
                        'auc': roc_auc_score(y_te, y_proba),
                        'n_features': X.shape[1]})

results_df = pd.DataFrame(results)
avg_auc = results_df.groupby('combination')['auc'].mean().sort_values(ascending=False)
best = avg_auc.index[0]
log.info(f"Best: {best} avg AUC={avg_auc.iloc[0]:.4f}")

2026-06-26 21:11:29,104 INFO Evaluating aa


2026-06-26 21:11:42,491 INFO Evaluating kmer2


2026-06-26 21:12:07,882 INFO Evaluating kmer3


2026-06-26 21:12:35,620 INFO Evaluating onehot


2026-06-26 21:12:48,638 INFO Evaluating physicochemical


...

2026-06-26 21:15:27,099 INFO Evaluating aa+kmer2+onehot+physicochemical+blosum62+length_moments


2026-06-26 21:15:53,940 INFO Evaluating aa+kmer2+onehot+physicochemical+blosum62+length_moments+genome_dna


2026-06-26 21:16:21,006 INFO Evaluating aa+genome_dna


2026-06-26 21:16:35,148 INFO Evaluating genome_dna


2026-06-26 21:16:48,368 INFO Best: aa+kmer2 avg AUC=0.6564


In [8]:
print(avg_auc.rename('avg_AUC').round(4).sort_values(ascending=False).to_string())
results_df.to_csv(DATA_DIR / 'feature_evaluation_results.csv', index=False)
with open(DATA_DIR / 'best_feature_combination.json', 'w') as f:
    json.dump({'combination': best, 'avg_auc': avg_auc.iloc[0]}, f)

combination
aa+kmer2                                                              0.6564
aa+physicochemical+onehot+kmer2+kmer3                                 0.6542
aa+kmer2+onehot+physicochemical+blosum62+length_moments               0.6512
aa+physicochemical+blosum62+length_moments                            0.6496
aa+onehot                                                             0.6478
kmer2                                                                 0.6464
aa                                                                    0.6455
onehot                                                                0.6455
aa+physicochemical                                                    0.6438
physicochemical+blosum62+length_moments                               0.6409
blosum62                                                              0.6354
physicochemical                                                       0.6103
kmer3                                                           